## Learning-to-Rank (LTR) Model Training with LightGBM

In the previous notebook, we generated LTR feature datasets containing user_id, item_id, label (implicit positive signals), group_id (user-level grouping), ALS user & item embeddings (dense features). This notebook trains a **Listwise LightGBM LTR model** using the features produced earlier.

### Agenda

1. We will first load LTR datasets  
2. Then, we will prepare LightGBM ranking datasets  
3. After this, we will train a **Listwise LightGBM model**  
4. Next, we will evaluate ranking quality using **NDCG@10**  
5. Finally, we wil export model + predictions + metrics

### Key Takeaways

From this exercise, we will learn:
- How to train a *listwise* ranking model using LightGBM’s ranking mode  
- How user-level grouping controls the ranking objective  
- How to compute NDCG@10 for validation and test splits  
- How LTR models convert rich features into improved ranking performance

### Setup

In [1]:
! pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 45.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from tqdm import tqdm
import json
from pathlib import Path

### Load LTR Feature Datasets

In [3]:
base_path = "/content/"
train = pd.read_parquet(base_path + "ltr_train.parquet")
valid = pd.read_parquet(base_path + "ltr_valid.parquet")
test  = pd.read_parquet(base_path + "ltr_test.parquet")

train.head()

,user_id,item_id,label,group_id,user_vec_0,user_vec_1,user_vec_2,user_vec_3,user_vec_4,user_vec_5,...,item_vec_54,item_vec_55,item_vec_56,item_vec_57,item_vec_58,item_vec_59,item_vec_60,item_vec_61,item_vec_62,item_vec_63
0,1,3186,1,1,-0.614307,-0.033299,-0.257872,-0.892903,0.941784,-0.660275,...,0.015741,0.010561,0.009570,0.018333,0.010598,0.022957,0.035207,0.022774,-0.008401,0.013939
1,1,1270,1,1,-0.614307,-0.033299,-0.257872,-0.892903,0.941784,-0.660275,...,-0.041802,0.037056,-0.050980,-0.005331,-0.020391,-0.011028,0.009913,0.030588,-0.036800,0.026942
2,1,1721,1,1,-0.614307,-0.033299,-0.257872,-0.892903,0.941784,-0.660275,...,-0.041548,-0.042661,0.062999,0.023627,0.042450,0.009877,0.043639,0.080892,-0.023868,0.034969
3,1,1022,1,1,-0.614307,-0.033299,-0.257872,-0.892903,0.941784,-0.660275,...,-0.000674,0.012028,0.029629,-0.005381,0.051983,0.028424,0.015439,0.019080,-0.027850,0.020953
4,1,2340,0,1,-0.614307,-0.033299,-0.257872,-0.892903,0.941784,-0.660275,...,0.009313,0.015433,0.004966,0.013224,-0.007483,0.016334,0.029876,0.003552,0.016705,0.013595


### Identify Feature Columns

Next, let us decode the feature columns and specify:
- Labels: `label`
- Query groups: `group_id` (each user = 1 group)
- Features: all vector columns

In [4]:
feature_cols = [c for c in train.columns if c.startswith("user_vec_") or c.startswith("item_vec_")]

len(feature_cols), feature_cols[:5]

(128, ['user_vec_0', 'user_vec_1', 'user_vec_2', 'user_vec_3', 'user_vec_4'])

### Build LightGBM Ranking Datasets

In [5]:
def build_lgb_dataset(df):
    X = df[feature_cols].astype("float32")
    y = df["label"].astype("float32")

    # group sizes = number of rows per user
    group = df.groupby("group_id").size().tolist()

    return lgb.Dataset(X, label=y, group=group)

train_ds = build_lgb_dataset(train)
valid_ds = build_lgb_dataset(valid)

In [18]:
display(valid["label"].value_counts())
display(train["label"].value_counts())

,count
label,
1,3486
0,2554


,count
label,
1,568129
0,419708


### Train LightGBM Listwise Ranking Model

Now, we will train the LightGBM listwise ranking model.
We will use:
- **lambdarank** objective (listwise ranking)
- Metric = **NDCG@10**

In [6]:
params = {
    "objective": "lambdarank",
    "metric": "ndcg@10",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 40,
    "feature_pre_filter": False,
    "verbosity": -1,
}

In [7]:
model = lgb.train(
    params,
    train_set=train_ds,
    valid_sets=[valid_ds],
    num_boost_round=500,
)

In [8]:
model.save_model("lgb_ltr_model.txt")
print("Model trained & saved.")

Model trained & saved.


### Predict Scores for Validation & Test Splits


In [9]:
def rank_split(df):
    X = df[feature_cols].astype("float32")
    df["score"] = model.predict(X, num_iteration=model.best_iteration)
    return df

In [10]:
valid_scored = rank_split(valid.copy())
test_scored  = rank_split(test.copy())

valid_scored.head()

,user_id,item_id,label,group_id,user_vec_0,user_vec_1,user_vec_2,user_vec_3,user_vec_4,user_vec_5,...,item_vec_55,item_vec_56,item_vec_57,item_vec_58,item_vec_59,item_vec_60,item_vec_61,item_vec_62,item_vec_63,score
0,1,1907,1,1,-0.614307,-0.033299,-0.257872,-0.892903,0.941784,-0.660275,...,0.035324,0.037506,0.030908,0.052913,0.042610,0.013816,0.022025,-0.022743,-0.002286,-0.691495
1,2,1544,1,2,0.341550,1.196947,-0.273511,-0.185006,-0.542383,-1.115729,...,-0.000719,0.006573,0.017062,-0.001855,0.027127,-0.029364,0.006828,-0.012751,0.004337,-2.197651
2,3,3868,0,3,-0.170509,-0.320222,-0.380353,-0.620376,0.404573,-0.448113,...,0.013534,-0.026260,0.028348,0.016018,-0.011617,0.007313,0.007384,-0.004269,-0.016471,-0.652403
3,4,1036,1,4,0.271791,0.483175,0.412064,-0.117494,-0.062333,0.206825,...,-0.019571,0.020611,0.049267,-0.009473,-0.011970,-0.043242,0.073396,0.020621,0.101472,0.377480
4,5,1884,0,5,-1.375939,0.599988,0.948864,0.038125,-0.938332,-0.132830,...,-0.021034,-0.001142,0.019128,0.015895,0.024961,0.010973,0.001234,0.016808,-0.004671,-1.620285


In [19]:
test_scored.head()

,user_id,item_id,label,group_id,user_vec_0,user_vec_1,user_vec_2,user_vec_3,user_vec_4,user_vec_5,...,item_vec_55,item_vec_56,item_vec_57,item_vec_58,item_vec_59,item_vec_60,item_vec_61,item_vec_62,item_vec_63,score
0,1,48,1,1,-0.614307,-0.033299,-0.257872,-0.892903,0.941784,-0.660275,...,0.023940,0.018180,0.026278,0.038220,0.029338,0.007077,0.012416,-0.010736,0.009131,-1.433247
1,2,1917,0,2,0.341550,1.196947,-0.273511,-0.185006,-0.542383,-1.115729,...,0.010877,0.028983,-0.007509,-0.017383,-0.003231,-0.020691,-0.021241,-0.027823,0.059960,-1.892537
2,3,2081,1,3,-0.170509,-0.320222,-0.380353,-0.620376,0.404573,-0.448113,...,0.004789,0.011534,0.011328,0.040834,0.049105,0.001697,0.023815,-0.018418,0.011098,-0.866354
3,4,1954,1,4,0.271791,0.483175,0.412064,-0.117494,-0.062333,0.206825,...,0.000459,0.045233,0.044546,-0.000829,-0.030823,0.016073,0.067939,0.022784,0.014791,-0.616646
4,5,288,0,5,-1.375939,0.599988,0.948864,0.038125,-0.938332,-0.132830,...,-0.006189,0.012975,0.025299,0.010827,0.011161,0.002567,-0.004486,0.014984,0.023260,-1.774457


### Compute NDCG@10

We compute NDCG per-user, then average across users.

In [11]:
def ndcg_at_k(labels, scores, k=10):
    """Compute NDCG@k for a single user."""
    df = pd.DataFrame({"label": labels, "score": scores})
    df = df.sort_values("score", ascending=False).head(k)

    dcg = (df["label"] / np.log2(np.arange(2, len(df)+2))).sum()
    ideal = (df["label"].sort_values(ascending=False) / np.log2(np.arange(2, len(df)+2))).sum()
    return dcg / ideal if ideal > 0 else 0.0

In [12]:
def compute_ndcg(df, k=10):
    scores = []
    for gid, group in df.groupby("group_id"):
        scores.append(ndcg_at_k(group.label.values, group.score.values, k=k))
    return float(np.mean(scores))

In [13]:
ndcg_val  = compute_ndcg(valid_scored, k=10)
ndcg_test = compute_ndcg(test_scored,  k=10)

In [14]:
ndcg_val, ndcg_test

(0.5771523178807947, 0.5899006622516556)

### Saving the Predictions

In [ ]:
valid_scored.to_parquet("ltr_predictions_val.parquet")
test_scored.to_parquet("ltr_predictions_test.parquet")
print("Predictions saved.")

Predictions saved.


### Saving the metrics

In [15]:
metrics = {
    "ndcg@10_val": ndcg_val,
    "ndcg@10_test": ndcg_test,
}

with open("ltr_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

metrics

{'ndcg@10_val': 0.5771523178807947, 'ndcg@10_test': 0.5899006622516556}

### Conclusion

In this exercise, we trained a **Listwise LightGBM LTR model** on the
user–item interaction features generated in previous notebook. We computed **NDCG@10** on the validation and test splits,demonstrating how ranking models use group-level objectives to optimise the quality of top-N recommendations.